# Three-dimensional tetrahedral MHM

An exact affine patch followed by the archived five-level sinusoidal cube study. P2 local pressure and four independent P0 subtriangles per macroface. The raw gradient flux is broken; skeleton flux is macro conservative.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/04e2dac57de97149928d489b0fda19b4f7f54dcfbae98b2c74b6f91ae61e60d7/29_darcy3d-companion.zip"
COMPANION_SHA256 = "04e2dac57de97149928d489b0fda19b4f7f54dcfbae98b2c74b6f91ae61e60d7"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/29_darcy3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm.meshes.tetrahedron import TetraMesh
from pymhm.fem.traces.triangle_3d import TriangularSkeleton


## Declare local and global forms

The application defines the energy/source and both oriented trace incidences.
Qk Cartesian and Pk tetrahedral coordinates use their shared FEM kernels:

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T, & L_T(v)&=(f,v)_T,\\
b_T(\lambda,v)&=\langle\lambda,v\rangle_{\partial T}, &
c_T(p,\mu)&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The declared `LocalEquations` use C=-B.T, a constant pressure kernel and its
physical integral moment. The additional global `Equation` supplies the weak
Dirichlet pressure functional with its prescribed sign. Neumann data are
outward physical fluxes; a fully Neumann boundary uses an explicit pressure
integral constraint. Generic assembly and field interpretation are separate
calls. The sampled gradient flux remains distinct from the conservative trace.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.tetrahedral_darcy import (
    define_tetrahedral_darcy,
    recover_tetrahedral_darcy,
)


def pressure(points):
    """Return a nonhomogeneous affine pressure patch."""
    return 1 + points @ np.array([1.0, 2.0, 3.0])


mesh = TetraMesh.unit_cube(1)
solution_definition = define_tetrahedral_darcy(
    mesh, skeleton=TriangularSkeleton(mesh, 2), degree=2, local_refinement=2, dirichlet=pressure
)
solution_system = assemble(solution_definition.problem)
solution = recover_tetrahedral_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
assert solution.l2_error(pressure) < 1e-12
assert solution.flux_l2_error([-1.0, -2.0, -3.0]) < 1e-11
assert np.max(np.abs(solution.conservation_residuals())) < 1e-12
(solution.l2_error(pressure), solution.flux_l2_error([-1.0, -2.0, -3.0]))


## Recorded convergence

The PDE matches the sinusoidal cube example in Gomes et al. (2017), section 5. The Freudenthal macro meshes and red-refined local meshes differ from its TetGen meshes; these data verify convergence and do not reproduce cluster timing tables. Volume norms use the full P2 function and gradient.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    report = json.loads((root / "examples/results/darcy3d.json").read_text())
    assert len(report["rows"]) == 5
    assert not report["source_changed_during_run"]
    for row in report["rows"]:
        assert row["macro_balance_max"] < 1e-12
    [(r["macro_tetrahedra"], r["pressure_l2_error"], r["flux_l2_error"]) for r in report["rows"]]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/darcy3d/convergence.png")))
    display(Image(filename=str(root / "docs/figures/darcy3d/fields.png")))


The cross-section preserves independent cut-cell values and highlights the actual macro intersections. Flat shading samples each cut-cell centroid; it does not replace the volumetric norm integration. Recompute with `python -m examples.solve_darcy3d` and render with `python -m examples.plot_darcy3d`.
